In [1]:
from ase.io import read, write
import os
import shutil
import numpy as np
from ase import Atoms

def write_poscar_with_displacements(atoms, filename="POSCAR_displaced"):
    """Write a POSCAR with random Gaussian displacements (σ=0.1 Å) on atoms and cell."""
    a0 = 3.0  # Lattice constant in Angstroms
    # Add random displacements to atomic positions (Gaussian, μ=0, σ=0.1 Å)
    displacements = np.random.normal(0, 0.05*a0, (len(atoms), 3))
    atoms.positions += displacements

    # Add random strains to cell vectors (same distribution)
    cell_displacements = np.random.normal(0, 0.025*a0, (3, 3))
    atoms.cell += cell_displacements

    # Write to POSCAR format
    with open(filename, 'w') as f:
        f.write("Displaced Structure\n")  # Title
        f.write('1.10343677\n')

        # Write strained cell vectors
        for vector in atoms.cell:
            f.write(f'{vector[0]:20.16f} {vector[1]:20.16f} {vector[2]:20.16f}\n')

        # Atomic symbols and counts
        symbols = atoms.get_chemical_symbols()
        unique_symbols = sorted(set(symbols), key=symbols.index)
        counts = [symbols.count(s) for s in unique_symbols]
        f.write(' '.join(unique_symbols) + '\n')
        f.write(' '.join(map(str, counts)) + '\n')

        # Cartesian coordinates (no Selective Dynamics)
        f.write("Cartesian\n")
        for pos in atoms.positions:
            f.write(f'{pos[0]:20.16f} {pos[1]:20.16f} {pos[2]:20.16f}\n')


# SLURM job script template
slurm_template = """#!/bin/bash
#SBATCH --job-name=Nb2_{i}_{j}
#SBATCH --output=slurm_{i}_{j}.out
#SBATCH --error=slurm_{i}_{j}.err
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=5
#SBATCH --cpus-per-task=1
#SBATCH --time=8:00:00
#SBATCH --partition=regularshort
#SBATCH --mem-per-cpu=8G

# Load necessary modules (if required)
module load VASP

# Run the calculation
srun vasp_std
"""

# List of source files
source_files = [
    '/scratch/p301616/VASP_DFT_bcc_DB/niobium_NewGammaSurf_110/INPUTS_temp/INCAR',
    '/scratch/p301616/VASP_DFT_bcc_DB/niobium_NewGammaSurf_110/INPUTS_temp/POTCAR'
]


###########################M
# MAIN CODE
############################

# Read the XYZ file

ny = 20
nz = 12
perf_110 = read('/scratch/p301616/VASP_DFT_bcc_DB/niobium_NewGammaSurf_112/112_plane.xyz')

# Loop through the first 1000 items in primitive_xyz
for i in range(ny):
    for j in range(nz):
        direc = 'Nb_gammaSurf_' + str(i) + '_' + str(j)

        os.makedirs(direc, exist_ok=True)
        os.chdir(direc)
        primitive_xyz = perf_110.copy()

        # add the displacement
        primitive_xyz.cell[0][1] += i * primitive_xyz.cell[1][1]/ny
        primitive_xyz.cell[0][2] += j * primitive_xyz.cell[2][2]/nz

        # Write the POSCAR file with Selective Dynamics
        write_poscar_with_displacements(primitive_xyz, filename="POSCAR")
        
        # Get the current directory
        current_directory = os.getcwd()
        # copy INCAR and POTCAR files
        for source_file in source_files:
            shutil.copy(source_file, current_directory)

        # Write the SLURM job script
        with open('job_{}_{}.sh'.format(i,j), 'w') as f:
            f.write(slurm_template.format(i=i,j=j))

        # Submit the SLURM job
        os.system('sbatch job_{}_{}.sh'.format(i,j))

        print('job {} + {} submitted!'.format(i,j))
        os.chdir("../")

Submitted batch job 17962927
job 0 + 0 submitted!
Submitted batch job 17962928
job 0 + 1 submitted!
Submitted batch job 17962929
job 0 + 2 submitted!
Submitted batch job 17962930
job 0 + 3 submitted!
Submitted batch job 17962931
job 0 + 4 submitted!
Submitted batch job 17962932
job 0 + 5 submitted!
Submitted batch job 17962933
job 0 + 6 submitted!
Submitted batch job 17962934
job 0 + 7 submitted!
Submitted batch job 17962935
job 0 + 8 submitted!
Submitted batch job 17962936
job 0 + 9 submitted!
Submitted batch job 17962937
job 0 + 10 submitted!
Submitted batch job 17962938
job 0 + 11 submitted!
Submitted batch job 17962939
job 1 + 0 submitted!
Submitted batch job 17962940
job 1 + 1 submitted!
Submitted batch job 17962941
job 1 + 2 submitted!
Submitted batch job 17962942
job 1 + 3 submitted!
Submitted batch job 17962943
job 1 + 4 submitted!
Submitted batch job 17962944
job 1 + 5 submitted!
job 1 + 6 submitted!
Submitted batch job 17962945
job 1 + 7 submitted!
Submitted batch job 1796294